# Exploración del modelo de decisión crediticia

Este cuaderno reproduce los pasos clave del pipeline: carga del dataset de trabajo, análisis exploratorio y aplicación del scorecard a un solicitante de ejemplo.

**Pre-requisito:** ejecutar `python run_pipeline.py` en la terminal (o al menos las etapas 0 y 2) para generar `data/processed/loan_data_features.csv.gz` y el modelo en `outputs/`.

## 1. Carga de datos

In [ ]:
import numpy as np
import pandas as pd
import joblib

from src import config
from src.data_prep import get_model_matrix
from src.scoring import apply_decision_rules

df = pd.read_csv(config.PROCESSED_CSV_GZ, compression="gzip", low_memory=False)
print(f"Registros: {len(df):,} | Columnas: {df.shape[1]}")
print(f"Tasa de mal préstamo: {df['bad_loan'].mean():.2%}")

## 2. Análisis exploratorio

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(figsize=(8, 4.5))
order = ["A", "B", "C", "D", "E", "F", "G"]
rates = [df.loc[df["grade"] == g, "bad_loan"].mean() * 100 for g in order]
sns.barplot(x=order, y=rates, palette="Blues_d", ax=ax)
ax.set_title("Tasa de mal préstamo por calificación (grade)")
ax.set_xlabel("Grade")
ax.set_ylabel("Tasa de mal préstamo (%)")
plt.show()

## 3. Aplicación del modelo a un solicitante de ejemplo

Cargamos el modelo entrenado y decidimos sobre una solicitud hipotética.

In [ ]:
model = joblib.load(config.OUTPUTS / "modelo_logistico.joblib")
rules = pd.read_json(config.OUTPUTS / "reglas_decision.json", typ="series")

In [ ]:
# Solicitante de ejemplo: perfil conservador (buena calificación)
ejemplo = df.iloc[[42]].copy()
X = get_model_matrix(ejemplo)
prob = model.predict_proba(X)[:, 1]
pred = apply_decision_rules(prob, {
    "score_cutoff": rules["score_cutoff"],
    "score_interno": rules["score_interno"],
    "scorecard": {"factor": rules["factor_scorecard"], "offset": rules["offset_scorecard"]},
})
print(f"Score: {pred['score'].iloc[0]:.1f}")
print(f"Decisión: {pred['decision'].iloc[0]}")
print(f"Segmento: {pred['segmento'].iloc[0]}")

## 4. Reglas de decisión

- `score >= 600` → **Aprobado** (25% de menor riesgo).
- `score >= 613` (dentro de aprobados) → segmento **Bueno**; en caso contrario, segmento **Malo**.
- `score < 600` → **Rechazado**.